<a href="https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/08-operations/02-reliability-and-fallbacks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reliability & Fallbacks

**Goal:** Make an LLM feature survive the real world — rate limits, timeouts, transient errors, bad outputs, and a provider that's simply down — with retries, backoff, fallbacks, and graceful degradation.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks) — the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).

## Setup

Each notebook is self-contained, so the next two cells stand it up from scratch:

1. **Install dependencies.** The `aien` package (this repo) carries the shared setup helper and pulls in the `groq` client — the only dependency this notebook needs.
2. **Load your API key.** Free key at [console.groq.com](https://console.groq.com/) (no credit card); in Colab add it via the **key icon** → **Add new secret** named exactly `GROQ_API_KEY`, notebook access on. Locally, set `GROQ_API_KEY` in your environment.

(Full walkthrough: [00-setup/00-environment.ipynb](https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/00-setup/00-environment.ipynb).)

In [ ]:
%pip install -q "git+https://github.com/calmrocks/ai-engineer-notebooks.git"

In [ ]:
from aien import setup

# Loads GROQ_API_KEY and returns a ready Groq client.
client, MODEL = setup()

## The model call is the least reliable thing in your stack

A database query fails rarely and fast. An LLM API call is the opposite: it's slow, rate-limited, occasionally errors transiently, sometimes returns malformed output, and every so often the whole provider has a bad hour. If your feature treats the call as if it always succeeds quickly, it will be your least reliable component.

Reliability engineering for LLMs is mostly the same toolkit you'd use for any flaky remote dependency — **retries, backoff, timeouts, fallbacks, circuit breakers, graceful degradation** — plus one LLM-specific twist: a call can "succeed" (HTTP 200) yet return output your app can't use. We'll build each layer on Groq.

## Layer 1: retry with exponential backoff + jitter

Rate limits (429) and transient 5xx errors are *expected*, not exceptional — the correct response is to wait and retry, not to fail. Back off exponentially so you don't hammer a struggling service, and add **jitter** so many clients retrying at once don't synchronize into a thundering herd. Retry only *retryable* errors; fail fast on the rest (a 400 won't fix itself).

In [ ]:
import time, random

# Groq raises typed errors; we retry the transient ones. (argless random is fine here --
# this runs live, not in a deterministic workflow.)
try:
    from groq import RateLimitError, APIStatusError, APIConnectionError
    RETRYABLE = (RateLimitError, APIConnectionError)
except Exception:                      # keep the notebook robust to SDK version drift
    RETRYABLE = ()

def call_with_retry(messages, max_attempts=4, base=0.5, **kwargs):
    for attempt in range(1, max_attempts + 1):
        try:
            return client.chat.completions.create(model=kwargs.pop("model", MODEL),
                                                   messages=messages, **kwargs)
        except RETRYABLE as e:
            if attempt == max_attempts:
                raise
            delay = base * (2 ** (attempt - 1)) + random.uniform(0, base)  # backoff + jitter
            print(f"  attempt {attempt} hit {type(e).__name__}; retrying in {delay:.2f}s")
            time.sleep(delay)

resp = call_with_retry([{"role": "user", "content": "Say 'reliable' and nothing else."}],
                       max_tokens=10)
print(resp.choices[0].message.content)

On a healthy connection this returns on attempt 1. Under rate limiting you'd see the retry lines fire and the call still succeed. The pattern — retry transient, backoff, jitter, cap attempts, re-raise at the end — is the single highest-value reliability habit; put it around every model call.

## Layer 2: timeouts and a fallback model

A call that hangs is worse than one that fails — it ties up resources and a waiting user. Set a **timeout**, and when the primary path fails (times out, exhausts retries, or the model is unavailable), **fall back**: to a smaller/faster model, a cheaper provider, or a cached/canned response. A degraded answer beats a spinner or a 500.

In [ ]:
def answer_with_fallback(question, timeout=10):
    # Primary: the default model. Fallback: a smaller one. In a real app the fallback
    # might be a different PROVIDER entirely, so one vendor's outage isn't your outage.
    chain = [MODEL, "openai/gpt-oss-20b"]
    for i, model in enumerate(chain):
        try:
            resp = call_with_retry(
                [{"role": "user", "content": question}],
                model=model, max_tokens=120, timeout=timeout,
            )
            return {"answer": resp.choices[0].message.content, "served_by": model,
                    "degraded": i > 0}
        except Exception as e:
            print(f"  {model} failed ({type(e).__name__}); "
                  f"{'falling back' if i+1 < len(chain) else 'no fallback left'}")
    # Last resort: never leave the user with nothing.
    return {"answer": "Sorry — I can't answer right now. Please try again shortly.",
            "served_by": None, "degraded": True}

print(answer_with_fallback("What is a subnet mask?"))

Normally the primary serves it (`degraded: False`). If it failed, you'd transparently drop to the smaller model, and only if *everything* fails does the user get the honest canned message — never an unhandled exception. The `degraded` flag is important: emit it as a metric (section 01) so you *know* how often you're running degraded, even when users don't notice.

## Layer 3: validate the output, then retry or degrade (the LLM-specific one)

> **⚠️ Production reality —** the failure mode ordinary services don't have: a **200 OK with unusable content.** The model returns invalid JSON, an off-list enum, an empty answer — the HTTP call "succeeded," but your app can't use the result. Treat it as a failure: validate, then retry once (the next sample is often fine), then degrade.

This is the reliability sibling of the schema-validation you did in `01-model-apis/01-structured-output` — same check, now wired into the retry/degrade machinery.

In [ ]:
import json, re

def get_structured(question, max_tries=3):
    prompt = (f'Answer as JSON: {{"answer": "<one sentence>", '
              f'"confidence": <0-1 number>}}. Question: {question}')
    for attempt in range(1, max_tries + 1):
        resp = call_with_retry([{"role": "user", "content": prompt}], max_tokens=120)
        raw = resp.choices[0].message.content
        try:
            obj = json.loads(re.search(r'\{.*\}', raw, re.DOTALL).group())
            assert isinstance(obj.get("answer"), str) and obj["answer"]
            assert isinstance(obj.get("confidence"), (int, float)) and 0 <= obj["confidence"] <= 1
            return obj                              # valid -> done
        except Exception:
            print(f"  attempt {attempt}: output failed validation, retrying")
    return {"answer": "unavailable", "confidence": 0.0, "degraded": True}  # safe default

print(get_structured("What layer of the OSI model is TCP?"))

Two distinct retry loops now stack: `call_with_retry` handles *transport* failures (429/5xx), and this loop handles *content* failures (bad shape). They're different problems with the same shape — retry, cap, then degrade to a safe default your app can always handle.

## Layer 4: circuit breaker — stop hammering a dead dependency

When a provider is genuinely down, retrying every request just burns latency and money and delays the fallback. A **circuit breaker** notices a run of failures, "opens" (fails fast straight to the fallback for a cool-off window), then "half-opens" to test if the service recovered. It's the difference between a provider outage costing you *seconds* per request versus your full retry budget per request.

In [ ]:
class CircuitBreaker:
    def __init__(self, threshold=5, cooldown_s=30):
        self.threshold, self.cooldown = threshold, cooldown_s
        self.fails, self.opened_at = 0, None
    def allow(self, now):
        if self.opened_at is None:
            return True                              # closed: normal
        if now - self.opened_at >= self.cooldown:
            self.opened_at = None; self.fails = 0    # half-open: give it a chance
            return True
        return False                                 # open: fail fast
    def record(self, ok, now):
        if ok:
            self.fails = 0; self.opened_at = None
        else:
            self.fails += 1
            if self.fails >= self.threshold:
                self.opened_at = now                 # trip

# Simulated: 8 calls against a "dead" provider. Watch the breaker trip and fail fast.
cb, clock = CircuitBreaker(threshold=3, cooldown_s=30), 0.0
for i in range(8):
    clock += 1
    if not cb.allow(clock):
        print(f"call {i}: circuit OPEN -> skip provider, go straight to fallback")
        continue
    ok = False                                       # pretend every real call fails
    cb.record(ok, clock)
    print(f"call {i}: tried provider, {'ok' if ok else 'FAILED'} (fails={cb.fails})")

Once three calls fail, the breaker opens and subsequent calls skip the dead provider entirely — instant fallback instead of waiting out the full retry budget every time. After the cooldown it half-opens to probe for recovery. In production the breaker's open/closed state is also a **metric worth alerting on** (section 01): an open breaker means you're running degraded.

## The reliability checklist

Wrap every production model call in these, roughly in this order:

1. **Retry transient errors** (429/5xx/connection) with exponential backoff + jitter; fail fast on 4xx.
2. **Timeout** every call — a hang is worse than a failure.
3. **Fall back** on exhaustion — smaller model, different provider, cache, or an honest canned response. Never a raw exception to the user.
4. **Validate the output**; a 200 with unusable content is a failure — retry then degrade.
5. **Circuit-break** a persistently failing dependency so an outage costs seconds, not your whole retry budget, per request.
6. **Emit `degraded` as a metric** (section 01) so you know how often the safety nets are catching you.

None of this is LLM-specific except layer 4's output validation — it's classic distributed-systems hygiene applied to the flakiest dependency in your stack. Which is exactly the point: an FDE who treats the model call with the same rigor as any remote dependency ships things that survive contact with production.

## Exercises

1. **Make it actually retry.** Force a rate limit (a tight loop of parallel calls on the free tier) and confirm `call_with_retry` recovers where a bare call would crash. Log how many attempts it took.
2. **Provider-diverse fallback.** Change the fallback chain so the second entry is a *different provider's* SDK (even stubbed). Why does same-provider fallback fail to protect you from the most important outage — the provider itself being down?
3. **Budget the retries.** Retries multiply cost and latency. For a p95 latency SLO of 3s and a per-call p50 of 0.8s, how many retries can you afford before you must give up and degrade? Tie it to the latency metric from `08-operations/01`.
4. **Combine everything.** Wrap `get_structured` in the circuit breaker and the fallback chain, and trace it with the tracer from notebook 01. You now have one function that retries transport, validates content, degrades gracefully, breaks on outage, and is fully observable — the shape of a real production LLM call.